# Passo 2 – Limpeza das bases e extração dos boatos

**Pré-requisito:** o `01_bases.ipynb` já foi executado (as bases estão em `TCC/dados/brutos` e a tabela `bases_unificadas.csv` existe).

Este notebook:

- **Parte A – Fake.Br:** remove textos vazios, curtos e duplicados. Se um texto de um par sai, o par inteiro sai.
- **Parte B – FactPolCheckBr:** extrai das checagens o texto original do boato e prepara os lotes de revisão.
- **Parte C – FakeRecogna:** remove linha sem rótulo, textos vazios, curtos e duplicados.
- Registra tudo que foi removido, por motivo e classe. Essa tabela vira o Quadro 1 do TCC.

Rótulo: `1` = falso, `0` = verdadeiro.

## 0. Preparação

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')

PASTAS = {
    'brutos': os.path.join(PASTA_TCC, 'dados', 'brutos'),
    'processados': os.path.join(PASTA_TCC, 'dados', 'processados'),
    'resultados': os.path.join(PASTA_TCC, 'resultados'),
    'revisao': os.path.join(PASTA_TCC, 'resultados', 'revisao'),
}
for caminho in PASTAS.values():
    os.makedirs(caminho, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option('display.max_colwidth', 100)

def normaliza(serie):
    """Chave para achar textos repetidos: minúsculas, sem pontuação, espaços únicos."""
    return (serie.fillna('').astype(str).str.lower()
            .str.replace(r'\W+', ' ', regex=True).str.strip())

def n_palavras(serie):
    return serie.fillna('').astype(str).str.split().str.len()

exclusoes = []   # registro de tudo que foi removido
def registrar(base, motivo, df_removido, coluna_classe='rotulo'):
    contagem = df_removido[coluna_classe].map({1: 'falso', 0: 'verdadeiro'}).fillna('sem rótulo').value_counts()
    for classe, n in contagem.items():
        exclusoes.append({'base': base, 'motivo': motivo, 'classe': classe, 'n': int(n)})

bases = pd.read_csv(os.path.join(PASTAS['processados'], 'bases_unificadas.csv'))
print(bases['base'].value_counts().to_string())

## Parte A – Fake.Br (versão de tamanho normalizado)

**Limite mínimo de palavras:** definido pelos dados, no **1º percentil** da distribuição de tamanhos. Abaixo disso, o texto é curto demais para ter conteúdo (títulos soltos, legendas de fotos).

**Regra dos pares:** cada notícia falsa tem uma verdadeira correspondente. Se um texto sai, o par inteiro sai. Assim as classes continuam equilibradas e a divisão por pares (Passo 4) continua possível.

In [ ]:
fakebr = bases[bases['base'] == 'Fake.Br'].copy().reset_index(drop=True)
fakebr['palavras'] = n_palavras(fakebr['texto'])
fakebr['chave'] = normaliza(fakebr['texto'])

LIMITE_FAKEBR = int(np.percentile(fakebr['palavras'], 1))
print('Textos antes da limpeza:', len(fakebr))
print('Limite mínimo (1º percentil):', LIMITE_FAKEBR, 'palavras')

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(fakebr['palavras'].clip(upper=600), bins=60, color='#4C72B0')
ax.axvline(LIMITE_FAKEBR, color='red', linestyle='--', label=f'limite = {LIMITE_FAKEBR} palavras')
ax.set_xlabel('Palavras por texto (cortado em 600)'); ax.set_ylabel('Textos'); ax.legend()
ax.set_title('Fake.Br: distribuição do tamanho dos textos')
plt.tight_layout()
plt.savefig(os.path.join(PASTAS['resultados'], 'distribuicao_tamanho_fakebr.png'), dpi=150)
plt.show()

In [ ]:
motivo = pd.Series('', index=fakebr.index)
motivo[fakebr['chave'] == ''] = 'vazio'
motivo[(motivo == '') & (fakebr['palavras'] < LIMITE_FAKEBR)] = 'curto'

restantes = fakebr[motivo == '']
motivo[restantes.index[restantes.duplicated('chave', keep='first')]] = 'duplicado'

pares_ruins = set(fakebr.loc[motivo != '', 'par_id'])
motivo[(motivo == '') & fakebr['par_id'].isin(pares_ruins)] = 'par removido'

for m in ['vazio', 'curto', 'duplicado', 'par removido']:
    registrar('Fake.Br', m, fakebr[motivo == m])

fakebr_limpo = fakebr[motivo == ''].drop(columns=['chave', 'palavras'])

# Conferências de segurança
assert fakebr_limpo.groupby('par_id').size().eq(2).all(), 'há par incompleto'
assert fakebr_limpo['rotulo'].value_counts().nunique() == 1, 'classes desequilibradas'

fakebr_limpo.to_csv(os.path.join(PASTAS['processados'], 'fakebr_limpo.csv'), index=False)
print('Removidos por motivo:'); print(motivo[motivo != ''].value_counts().to_string())
print('\nFake.Br limpo:', len(fakebr_limpo), 'textos |', fakebr_limpo['par_id'].nunique(), 'pares')
print(fakebr_limpo['rotulo'].map({1: 'falso', 0: 'verdadeiro'}).value_counts().to_string())

## Parte C – FakeRecogna

Mesma limpeza, sem pares. O limite de palavras também vem do 1º percentil.

Lembrete: o texto desta base já vem pré-processado pelos autores. Esta base só será usada no experimento de viés.

In [ ]:
recogna = bases[bases['base'] == 'FakeRecogna'].copy().reset_index(drop=True)
recogna['palavras'] = n_palavras(recogna['texto'])
recogna['chave'] = normaliza(recogna['texto'])
LIMITE_RECOGNA = int(np.percentile(recogna['palavras'], 1))
print('Textos antes da limpeza:', len(recogna), '| limite (1º percentil):', LIMITE_RECOGNA, 'palavras')

motivo = pd.Series('', index=recogna.index)
motivo[recogna['rotulo'].isna()] = 'sem rótulo'
motivo[(motivo == '') & (recogna['chave'] == '')] = 'vazio'
motivo[(motivo == '') & (recogna['palavras'] < LIMITE_RECOGNA)] = 'curto'
restantes = recogna[motivo == '']
motivo[restantes.index[restantes.duplicated('chave', keep='first')]] = 'duplicado'

for m in ['sem rótulo', 'vazio', 'curto', 'duplicado']:
    registrar('FakeRecogna', m, recogna[motivo == m])

recogna_limpo = recogna[motivo == ''].drop(columns=['chave', 'palavras', 'par_id'])
recogna_limpo['rotulo'] = recogna_limpo['rotulo'].astype(int)
recogna_limpo.to_csv(os.path.join(PASTAS['processados'], 'recogna_limpo.csv'), index=False)
print('Removidos por motivo:'); print(motivo[motivo != ''].value_counts().to_string())
print('\nFakeRecogna limpo:', len(recogna_limpo), 'textos')
print(recogna_limpo['rotulo'].map({1: 'falso', 0: 'verdadeiro'}).value_counts().to_string())

## Parte B – FactPolCheckBr: extrair o texto original do boato

O campo `texto` desta base é a **matéria da agência**, e não o boato. Mas muitas matérias reproduzem o boato como ele circulou. Este bloco tenta recuperar esse trecho com regras simples.

**Regras:**
1. **Boatos.org:** o trecho depois de "Versão 1:" (ou "Confira:" / "Leia a mensagem:"), até o próximo marcador.
2. **Citação entre aspas:** a maior citação com pelo menos 15 palavras, desde que o texto logo antes indique que ela é o conteúdo que circulou ("circula", "mensagem", "legenda", "áudio"…) e que não seja resposta de assessoria, nota oficial, norma ou o próprio desmentido.

O que as regras não pegam (boatos só em imagem ou vídeo, ou parafraseados) é descartado e contado. **Toda extração passa por revisão** antes de entrar no teste.

In [ ]:
import re

MIN_PALAVRAS = 15
MAX_PALAVRAS = 400

# Marcadores que encerram o trecho do boato no Boatos.org
FIM_BOATOS = re.compile(
    r'(Vers[ãa]o\s*\d+\s*:|O que (diz|dizem)|Por que|Como (sabemos|verificamos)|'
    r'Conclus[ãa]o|Fonte[s]?:|Leia (mais|também)|Segundo (o|a) |No entanto|Mas (é|na verdade)|'
    r'A (equipe|reportagem) |Não (é verdade|há|existe)|É falso|Falso:)', re.I)

# Palavras antes da citação que indicam que ela É o conteúdo que circulou
SINAIS_BOATO = re.compile(
    r'circul|mensagem|publica[çc]|legenda|áudio|vídeo|postagem|\bpost\b|tuíte|tweet|print|'
    r'corrente|compartilh|viraliz|alega|afirma:|diz:|escreve|conte[úu]do|memorando', re.I)
# Palavras antes da citação que indicam resposta, nota ou fala de fonte (não é o boato)
SINAIS_FONTE = re.compile(
    r'em nota|nota |assessoria|procurad|respondeu|esclareceu|informou|explicou|justificou|declarou|'
    r'ao (UOL|Estad|Aos Fatos|Comprova|Lupa|AFP|g1|Globo)|à (Lupa|AFP|reportagem)|disse o|afirmou o|afirmou a|'
    r'em entrevista|em discurso|comunicado|ministro|presidente do|segundo o (TSE|IBGE|Datafolha|Ipec)|'
    r'norma|diretriz|determina|estabelece|plano de governo|programa de governo|decreto|miss[ãa]o|relat[óo]rio|na íntegra|a fala', re.I)
# Palavras que indicam que o trecho é o desmentido, não o boato
SINAIS_VEREDITO = re.compile(
    r'é falso|é falsa|são falso|é mentira|informação (falsa|mentirosa)|desinforma|fake news|não existe|'
    r'não há (registro|evid|prova)|é enganoso|é verdade que não|checagem|checamos|verific', re.I)

def n_palavras(s):
    return len(s.split())

def _limpa(s):
    return re.sub(r'\s+', ' ', s).strip(' \t\n\r"“”:-–—')

def extrair_boatos_org(texto):
    """Trecho após 'Versão 1:' (ou 'Confira:' / 'Leia a mensagem:') até o próximo marcador."""
    m = re.search(r'Vers[ãa]o\s*1\s*:\s*(.+)', texto, flags=re.S)
    if not m:
        m = re.search(r'(Confira|Leia[^.\n:]{0,80}|Veja|Assista)\s*:\s*(.+)', texto, flags=re.S)
        if not m:
            return None
        resto = m.group(2)
    else:
        resto = m.group(1)
    corte = FIM_BOATOS.search(resto)
    trecho = resto[:corte.start()] if corte and corte.start() > 40 else resto
    trecho = _limpa(trecho)
    return trecho or None

def _citacoes(texto):
    """Devolve (inicio, fim, conteudo) de cada citação, pareando aspas com cuidado."""
    achados = [(m.start(), m.end(), m.group(1)) for m in re.finditer(r'“([^”]{20,}?)”', texto)]
    # aspas retas: só usa se o número de aspas for par, para não parear errado
    retas = [m.start() for m in re.finditer(r'"', texto)]
    if len(retas) % 2 == 0:
        for i in range(0, len(retas), 2):
            a, b = retas[i], retas[i + 1]
            if b - a > 20:
                achados.append((a, b + 1, texto[a + 1:b]))
    return achados

def extrair_citacao_longa(texto):
    """Maior citação com pelo menos MIN_PALAVRAS palavras, que pareça conteúdo que circulou."""
    texto = re.sub(r'\s+', ' ', texto)
    boas = []
    for ini, fim, conteudo in _citacoes(texto):
        c = _limpa(conteudo)
        if n_palavras(c) < MIN_PALAVRAS:
            continue
        if re.match(r'^[\.\,\;]', conteudo.strip()):
            continue  # aspas pareadas errado
        antes = texto[max(0, ini - 160):ini]
        if SINAIS_FONTE.search(antes[-120:]):
            continue
        if not SINAIS_BOATO.search(antes):
            continue
        if SINAIS_VEREDITO.search(c[:250]):
            continue
        boas.append(c)
    return max(boas, key=n_palavras) if boas else None

def extrair_boato(texto, agencia):
    """Devolve (trecho, regra) ou (None, None)."""
    if not isinstance(texto, str):
        return None, None
    if agencia == 'Boatos.org':
        t = extrair_boatos_org(texto)
        if t and n_palavras(t) >= MIN_PALAVRAS:
            return ' '.join(t.split()[:MAX_PALAVRAS]), 'versao_boatos'
    t = extrair_citacao_longa(texto)
    if t:
        return ' '.join(t.split()[:MAX_PALAVRAS]), 'citacao_aspas'
    return None, None


In [ ]:
fpc = pd.read_csv(os.path.join(PASTAS['brutos'], 'Dataset-FactPolCheckBr', 'dados', 'com_texto.csv'))
fpc = fpc.rename(columns={'Link': 'link', 'Título da checagem': 'titulo', 'Agência': 'agencia', 'texto': 'texto_checagem',
                          'Natureza da notícia': 'natureza'})
print('Checagens no arquivo:', len(fpc))

# 1) Só entram checagens classificadas como falsas pela agência
fora = fpc[fpc['natureza'] != 'Falsa'].copy()
for nome, g in fora.groupby(fora['natureza'].fillna('sem classificação')):
    exclusoes.append({'base': 'FactPolCheckBr', 'motivo': f'classificação: {nome}', 'classe': '-', 'n': len(g)})
fpc = fpc[fpc['natureza'] == 'Falsa'].copy()
print('Classificadas como falsas:', len(fpc))

# 2) Extração do boato
res = [extrair_boato(t, a) for t, a in zip(fpc['texto_checagem'], fpc['agencia'])]
fpc['trecho'] = [r[0] for r in res]
fpc['regra'] = [r[1] for r in res]

sem = fpc[fpc['trecho'].isna()]
exclusoes.append({'base': 'FactPolCheckBr', 'motivo': 'sem boato extraível', 'classe': 'falso', 'n': len(sem)})
fpc = fpc[fpc['trecho'].notna()].copy()

# 3) Mesmo boato citado em mais de uma checagem: fica uma só
fpc['chave'] = normaliza(fpc['trecho'])
dup = fpc.duplicated('chave', keep='first')
exclusoes.append({'base': 'FactPolCheckBr', 'motivo': 'boato repetido', 'classe': 'falso', 'n': int(dup.sum())})
fpc = fpc[~dup].drop(columns='chave').reset_index(drop=True)

print('Candidatos a boato:', len(fpc))
print(pd.crosstab(fpc['agencia'], fpc['regra'], margins=True).to_string())

### Lotes de revisão

Os candidatos são embaralhados com a semente 42 e divididos em lotes de 100. Cada lote é uma amostra aleatória e proporcional do conjunto. A coluna `contexto` mostra o que vem logo antes do trecho na matéria, para ajudar a decidir.

**Protocolo de decisão (para cada candidato):**
- **ok:** o trecho é o conteúdo do boato, como circulou (texto de mensagem, legenda, áudio transcrito, publicação).
- **corrigir:** o boato está presente, mas o trecho veio cortado ou misturado com outro texto. Informar o trecho certo.
- **descartar:** o trecho não é o boato. Exemplos: resposta de assessoria ou órgão, texto de lei ou plano de governo, fala verdadeira do político, trecho do desmentido, texto sem sentido sozinho.

In [ ]:
def contexto_antes(texto, trecho, n=140):
    t = re.sub(r'\s+', ' ', str(texto))
    i = t.find(trecho[:30])
    return t[max(0, i - n):i].strip() if i > 0 else ''

fpc['contexto'] = [contexto_antes(t, tr) for t, tr in zip(fpc['texto_checagem'], fpc['trecho'])]

fpc = fpc.sample(frac=1, random_state=SEMENTE).reset_index(drop=True)
fpc['id'] = ['fpc_%04d' % (i + 1) for i in range(len(fpc))]
fpc['lote'] = fpc.index // 100 + 1

# Arquivo completo para auditoria (sem o texto inteiro da matéria, que é pesado)
cols = ['id', 'lote', 'agencia', 'titulo', 'link', 'regra', 'contexto', 'trecho']
revisao = fpc[cols].copy()
revisao['decisao'] = ''; revisao['trecho_corrigido'] = ''; revisao['motivo'] = ''
revisao.to_csv(os.path.join(PASTAS['revisao'], 'revisao_completa.csv'), index=False, encoding='utf-8-sig')

# Lotes compactos: o que o revisor precisa ver
for n, g in revisao.groupby('lote'):
    g = g[['id', 'agencia', 'titulo', 'link', 'contexto', 'trecho']].copy()
    g['trecho'] = g['trecho'].str.split().str[:70].str.join(' ')
    g.to_csv(os.path.join(PASTAS['revisao'], f'lote_{n:02d}.csv'), index=False, encoding='utf-8-sig')

fpc.drop(columns=['texto_checagem']).to_csv(os.path.join(PASTAS['processados'], 'fpc_candidatos.csv'), index=False)
print('Candidatos:', len(fpc), '| lotes:', fpc['lote'].nunique())
print(fpc.groupby('lote').size().to_string())

## Quadro de exclusões (vai para a Seção 4.3)

In [ ]:
excl = pd.DataFrame(exclusoes)
excl.to_csv(os.path.join(PASTAS['resultados'], 'exclusoes.csv'), index=False)
print(excl.to_string(index=False))

quadro = pd.DataFrame({
    'Fake.Br': {'antes': (bases['base'] == 'Fake.Br').sum(), 'depois': len(fakebr_limpo)},
    'FakeRecogna': {'antes': (bases['base'] == 'FakeRecogna').sum(), 'depois': len(recogna_limpo)},
    'FactPolCheckBr (candidatos a boato)': {'antes': (bases['base'] == 'FactPolCheckBr').sum(), 'depois': len(fpc)},
}).T
quadro.to_csv(os.path.join(PASTAS['resultados'], 'quadro1_contagens.csv'))
quadro

## Próximo passo

1. Os lotes em `resultados/revisao/lote_XX.csv` são revisados seguindo o protocolo acima.
2. Depois da revisão, o notebook `02b_revisao.ipynb` lê as decisões, calcula a concordância com a sua conferência e monta o conjunto eleitoral definitivo.